# system_tai Phase 2 — exact KIS smoke test

This clean notebook runs text-query retrieval against the three audited videos. `DEVICE="auto"` uses CUDA when available and otherwise keeps CPU support. GPU selection changes latency, not ranking semantics or retrieval quality. Exact NumPy cosine retrieval remains the CPU correctness baseline. It writes only a manifest and JSONL checkpoint below `/kaggle/working/system_tai_outputs/kis_smoke/`. Image self-match compatibility is not evidence of text-retrieval quality; the Top-10 results below require manual visual inspection.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
REPO_ROOT = Path("/kaggle/working/AI_Challenge_HCM")
SYSTEM_ROOT = REPO_ROOT / "systems/system_tai"
OUTPUT_ROOT = Path("/kaggle/working/system_tai_outputs/kis_smoke")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_IDS = ["L21_V001", "L21_V002", "L22_V001"]
print(
    {"python": sys.version, "repository": SYSTEM_ROOT.is_dir(), "input_root": INPUT_ROOT.is_dir()}
)

## Install repository code only

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(SYSTEM_ROOT)], check=True)
INSTALL_OPENAI_CLIP = False
if INSTALL_OPENAI_CLIP:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "git+https://github.com/openai/CLIP.git",
        ],
        check=True,
    )
# Enable installation/download only under the Kaggle internet policy.

## Resolve one dataset root with bounded shallow discovery and build a manifest

In [ ]:
required_families = (
    Path("map-keyframes-aic25-b1/map-keyframes"),
    Path("clip-features-32-aic25-b1/clip-features-32"),
    Path("keyframes/keyframes"),
)
candidate_roots = sorted(
    path
    for path in (INPUT_ROOT / "datasets").glob("*/*")
    if path.is_dir()
)
valid_roots = [
    root
    for root in candidate_roots
    if all((root / relative).is_dir() for relative in required_families)
]
if len(valid_roots) != 1:
    raise RuntimeError(
        "expected exactly one Dataset_AIC2026 root under "
        f"{INPUT_ROOT / 'datasets'}/*/*; observed {len(valid_roots)}: "
        f"{[str(path) for path in valid_roots]}"
    )
dataset_root = valid_roots[0]
print(f"dataset root resolved: {dataset_root}")

artifacts_by_video = {}
for video_id in VIDEO_IDS:
    group = video_id.split("_", maxsplit=1)[0]
    artifacts = {
        "mapping_csv_path": dataset_root
        / "map-keyframes-aic25-b1/map-keyframes"
        / f"{video_id}.csv",
        "clip_npy_path": dataset_root
        / "clip-features-32-aic25-b1/clip-features-32"
        / f"{video_id}.npy",
        "keyframe_directory": dataset_root
        / "keyframes/keyframes"
        / f"Keyframes_{group}"
        / "keyframes"
        / video_id,
    }
    missing = [
        str(path)
        for name, path in artifacts.items()
        if not (path.is_dir() if name == "keyframe_directory" else path.is_file())
    ]
    if missing:
        raise FileNotFoundError(
            f"missing required artifacts for {video_id}: {missing}"
        )
    artifacts_by_video[video_id] = artifacts
manifest = {
    "videos": [
        {
            "video_id": video_id,
            "mapping_csv_path": str(
                artifacts_by_video[video_id]["mapping_csv_path"]
            ),
            "clip_npy_path": str(
                artifacts_by_video[video_id]["clip_npy_path"]
            ),
        }
        for video_id in VIDEO_IDS
    ]
}
manifest_path = OUTPUT_ROOT / "feature_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"manifest created: {manifest_path}")

## Load all 867 audited feature rows and initialize the canonical encoder

In [ ]:
import torch

from system_tai.common.schemas import KISQuery
from system_tai.features.btc_clip_store import FeatureStoreRegistry
from system_tai.features.query_encoder import OpenAIClipTextEncoder
from system_tai.retrieval.vector_search import ExactNumpyRetriever

DEVICE = "auto"  # supported values: auto, cpu, cuda
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError(f"unsupported DEVICE value: {DEVICE}")
CUDA_AVAILABLE = torch.cuda.is_available()
if DEVICE == "auto":
    selected_device = "cuda" if CUDA_AVAILABLE else "cpu"
elif DEVICE == "cuda" and not CUDA_AVAILABLE:
    raise RuntimeError(
        "DEVICE='cuda' was requested but CUDA is unavailable. Enable a Kaggle "
        "GPU accelerator or configure DEVICE='cpu'."
    )
else:
    selected_device = DEVICE
GPU_NAME = torch.cuda.get_device_name(0) if selected_device == "cuda" else None
print(f"selected device: {selected_device}")
print(f"CUDA available: {CUDA_AVAILABLE}")
if GPU_NAME is not None:
    print(f"GPU name: {GPU_NAME}")
print("device selection affects latency, not retrieval semantics or ranking quality")

registry = FeatureStoreRegistry.from_manifest(
    manifest_path, expected_dimension=512, memory_map=True
)
assert registry.total_rows == 867, registry.total_rows
print(f"registry loaded with row count: {registry.total_rows}")
ALLOW_MODEL_DOWNLOAD = False
encoder = OpenAIClipTextEncoder(
    device=selected_device, allow_model_download=ALLOW_MODEL_DOWNLOAD
)
print(f"text encoder loaded: device={encoder.identifiers['device']}")
retriever = ExactNumpyRetriever(registry, encoder, chunk_size=4096)
print(
    f"exact NumPy retriever ready for {len(registry.stores)} videos "
    "(CPU cosine baseline)"
)

## Direct Vietnamese diagnostic

The original Vietnamese queries are retained as negative/diagnostic evidence.

## English-translated diagnostic

The paired English translations are evaluated separately. Translation must not be treated as automatically successful.

In [ ]:
from system_tai.ranking.kis_ranker import (
    KISRanker,
    TemporalSuppressionConfig,
)

direct_vietnamese_queries = [
    ("vi_01", "một người đi xe máy dưới trời mưa lớn"),
    ("vi_02", "một chiếc ô tô dừng trước đèn giao thông"),
    ("vi_03", "nhiều người đang đi bộ trên đường phố"),
]
english_translated_queries = [
    ("vi_01_en", "a person riding a motorcycle in heavy rain"),
    ("vi_02_en", "a car stopped at a traffic light"),
    ("vi_03_en", "many people walking on a city street"),
]
query_groups = [
    ("direct Vietnamese diagnostic", direct_vietnamese_queries),
    ("English-translated diagnostic", english_translated_queries),
]
DIVERSIFY_TOP10_DISPLAY = False
DISPLAY_MINIMUM_FRAME_GAP = 90
DISPLAY_MAXIMUM_CANDIDATES_PER_VIDEO = None
display_ranker = KISRanker()
top100_results = []
top10_display_results = []
for group_label, queries in query_groups:
    print(f"\n=== {group_label} ===")
    for query_id, text in queries:
        print(f"query started: {query_id}")
        result = retriever.retrieve(
            KISQuery(query_id=query_id, text=text, top_k=100)
        )
        top100_results.append(result)
        display_result, suppression_report = display_ranker.apply(
            result,
            TemporalSuppressionConfig(
                enabled=DIVERSIFY_TOP10_DISPLAY,
                minimum_frame_gap=DISPLAY_MINIMUM_FRAME_GAP,
                maximum_candidates_per_video=(
                    DISPLAY_MAXIMUM_CANDIDATES_PER_VIDEO
                ),
            ),
        )
        top10_display_results.append(display_result)
        print(
            f"query completed: {query_id}; exact_results="
            f"{len(result.ranked_candidates)}"
        )
        print(
            "diversified diagnostic output: "
            f"enabled={suppression_report.enabled}; "
            f"removed={suppression_report.removed_count}"
        )
        print(query_id, text)
        for item in display_result.ranked_candidates[:10]:
            print(
                {
                    "video_id": item.video_id,
                    "frame_id": item.frame_id,
                    "clip_row": item.clip_row,
                    "score": round(item.score, 6),
                }
            )
# Scores are diagnostic; manual image review is required.

## Optional keyframe display from `/kaggle/input`

This cell reads images in place and never copies them. It resolves the displayed candidate by `keyframe_order`; it never treats a filename or row number as `frame_id`.

In [ ]:
SHOW_KEYFRAMES = False
if SHOW_KEYFRAMES:
    from IPython.display import Image, display

    image_extensions = {".jpg", ".jpeg", ".png"}
    for result in top10_display_results:
        print(result.query_id)
        for candidate in result.ranked_candidates[:3]:
            keyframe_root = artifacts_by_video[candidate.video_id][
                "keyframe_directory"
            ]
            matches = [
                path
                for path in keyframe_root.iterdir()
                if path.is_file()
                and path.suffix.lower() in image_extensions
                and path.stem.isdigit()
                and int(path.stem) == candidate.keyframe_order
            ]
            print(candidate.video_id, candidate.frame_id, candidate.score)
            if len(matches) == 1:
                display(Image(filename=str(matches[0]), width=320))
            else:
                print("Expected one image, observed", len(matches))

## Export Top-100 per query and validate the proposed shared checkpoint

In [ ]:
from system_tai.checkpointing.exporter import CheckpointExporter
from system_tai.validation.checkpoint_validator import CheckpointValidator

checkpoint_path = OUTPUT_ROOT / "diagnostic_queries_top100.jsonl"
export_summary = CheckpointExporter().export(top100_results, checkpoint_path)
print(f"JSONL written: {checkpoint_path}; records={export_summary.record_count}")
validation = CheckpointValidator().validate(checkpoint_path, registry=registry)
print(f"validator result: valid={validation.valid}; errors={len(validation.errors)}")
assert validation.valid, validation.errors
print(
    {
        "export": export_summary,
        "validation": validation,
        "manual_quality_review_required": True,
    }
)